In [ ]:
pip install yfinance

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm


def obtener_variacion_logaritmica_porcentaje(ticker):
    """
    Esta función devuelve un DataFrame con las variaciones logarítmicas de los precios de cierre
    de un ticker dado para un rango de fechas, expresadas en porcentaje y con el formato decimal ajustado
    para usar comas como separadores decimales, además de añadir el símbolo de porcentaje.

    Parámetros:
    - ticker: El símbolo del ticker de la acción (como string).
    - fecha_inicio: La fecha de inicio del rango en formato 'AAAA-MM-DD'.
    - fecha_fin: La fecha de fin del rango en formato 'AAAA-MM-DD'.
    """
    fecha_inicio = "2020-01-01"
    fecha_fin = "2023-12-31"
    # Descargar los datos del ticker
    datos = yf.download(ticker, start=fecha_inicio, end=fecha_fin)

    # Seleccionar solo la columna 'Close'
    precios_cierre = datos[['Close']]

    # Calcular la variación logarítmica de los precios de cierre
    variacion_log = np.log(precios_cierre / precios_cierre.shift(1))

    # Convertir la variación logarítmica a formato porcentual
    variacion_log_porcentaje = variacion_log * 100

    # Convertir a string, usar coma como separador decimal y añadir el símbolo de porcentaje
    variacion_log_porcentaje = variacion_log_porcentaje['Close'].replace('.', ',')

    # Crear un nuevo DataFrame para devolver, usando la fecha como índice
    df_resultado = pd.DataFrame(variacion_log_porcentaje)
    df_resultado.rename(columns={'Close': 'Variacion Logaritmica (%)'}, inplace=True)

    return df_resultado



In [ ]:

def regresion(ticker):

  # Ejemplo de uso
  # Símbolo del ticker para Apple Inc.


  df_variacion_log = obtener_variacion_logaritmica_porcentaje(ticker)
  df_variacion_log.head()


  famafrench_df = pd.read_csv('/content/csv_general.csv', sep = ';')


  # Convierte la columna de fecha a datetime
  famafrench_df['Date'] = pd.to_datetime(famafrench_df['Date'])

  # Si necesitas un formato específico, puedes usar el parámetro 'format'
  # df['Fecha'] = pd.to_datetime(df['Fecha'], format='%Y-%m-%d')

  # Después de convertir, establece la columna de fecha como índice si deseas
  famafrench_df.set_index('Date', inplace=True)



  famafrench_df.head()

  # Verifica los nombres de las columnas de ambos DataFrames


  # Si 'Date' ya es el índice o si la columna tiene otro nombre, ajusta el código.
  # Supongamos que la fecha ya es el índice o tiene otro nombre, entonces puedes saltarte el paso de set_index

  # Si ambos DataFrames tienen el índice de fecha correctamente configurado, puedes proceder directamente a merge
  df_combinado = famafrench_df.merge(df_variacion_log, left_index=True, right_index=True, how='outer')
  # Eliminar filas que contengan algún valor NaN
  df_combinado = df_combinado.dropna()

  # Convertir el índice de fecha a una columna regular

  df_combinado = df_combinado.round(3)



  # Asegúrate de que las columnas son tratadas como strings antes de reemplazar ',' por '.'
  df_combinado['Mkt-RF'] = df_combinado['Mkt-RF'].astype(str).str.replace(',', '.').astype(float)
  df_combinado['SMB'] = df_combinado['SMB'].astype(str).str.replace(',', '.').astype(float)
  df_combinado['HML'] = df_combinado['HML'].astype(str).str.replace(',', '.').astype(float)


  # Asegura que Pandas trate las columnas como strings antes de realizar operaciones de strings
  df_combinado['RF'] = df_combinado['RF'].astype(str).str.rstrip('%').str.replace(',', '.').astype(float) / 100
  df_combinado['Variacion Logaritmica (%)'] = df_combinado['Variacion Logaritmica (%)'].astype(str).str.rstrip('%').str.replace(',', '.').astype(float) / 100

  # Para % Fundflows, asume que ya has manejado los NaNs o que están siendo manejados de manera implícita aquí
  df_combinado['% Fundflows'] = df_combinado['% Fundflows'].astype(str).str.rstrip('%').str.replace(',', '.').astype(float) / 100



  # Preparar las variables independientes
  # Añadir una constante al modelo para el término de intercepción
  X = df_combinado[['Mkt-RF', 'SMB', 'HML', '% Fundflows']]
  X = sm.add_constant(X)

  # La variable dependiente es el exceso de rendimiento de Apple
  # Asegúrate de que 'RF' y 'rendimientos apple' están en formato decimal adecuado
  Y = df_combinado['Variacion Logaritmica (%)'] - df_combinado['RF']

  # Estimar el modelo OLS
  modelo = sm.OLS(Y, X).fit()

  # Mostrar el resumen del modelo
  print(modelo.summary())
  return df_resultado



In [ ]:
# Asume que esta es tu lista de tickers
lista_tickers = ["MSFT",
"AAPL",
"NVDA",
"AMZN",
"META",
"GOOGL",
"GOOG",
"BRK.B",
"LLY",
"AVGO",
"JPM",
"TSLA",
"XOM",
"V",
"UNH",
"MA",
"PG",
"JNJ",
"HD",
"MRK",
"COST",
"ABBV",
"CRM",
"CVX",
"AMD",
"NFLX",
"BAC",
"WMT",
"PEP",
"KO",
"LIN",
"TMO",
"ADBE",
"DIS",
"ACN",
"WFC",
"ORCL",
"CSCO",
"MCD",
"QCOM",
"ABT",
"CAT",
"INTU",
"AMAT",
"IBM",
"VZ",
"GE",
"CMCSA",
"NOW",
"INTC",
"DHR",
"COP",
"UBER",
"TXN",
"PFE",
"UNP",
"AMGN",
"PM",
"LOW",
"SPGI",
"ISRG",
"MU",
"RTX",
"GS",
"NEE",
"HON",
"ETN",
"AXP",
"LRCX",
"BKNG",
"PGR",
"T",
"ELV",
"SYK",
"C",
"MS",
"PLD",
"BLK",
"MDT",
"TJX",
"NKE",
"UPS",
"SCHW",
"DE",
"CI",
"BA",
"VRTX",
"BMY",
"CB",
"ADP",
"MMC",
"BSX",
"REGN",
"SBUX",
"ADI",
"LMT",
"FI",
"KLAC",
"CVS",
"BX",
"MDLZ",
"AMT",
"SNPS",
"GILD",
"PANW",
"CDNS",
"TMUS",
"CMG",
"MPC",
"EOG",
"ICE",
"TGT",
"SHW",
"SLB",
"CME",
"SO",
"ZTS",
"WM",
"ANET",
"DUK",
"MO",
"EQIX",
"PH",
"PSX",
"CL",
"ITW",
"FCX",
"PYPL",
"CSX",
"BDX",
"MCK",
"ABNB",
"APH",
"TT",
"TDG",
"USB",
"GD",
"ORLY",
"EMR",
"HCA",
"NOC",
"PNC",
"PCAR",
"AON",
"FDX",
"PXD",
"NXPI",
"MAR",
"MCO",
"VLO",
"CEG",
"CTAS",
"MSI",
"ROP",
"ECL",
"NSC",
"EW",
"COF",
"AIG",
"DXCM",
"HLT",
"AZO",
"APD",
"F",
"TRV",
"AJG",
"ADSK",
"TFC",
"GM",
"WELL",
"MMM",
"NUE",
"SPG",
"CPRT",
"CARR",
"MCHP",
"URI",
"ROST",
"WMB",
"DHI",
"SMCI",
"OKE",
"PSA",
"NEM",
"OXY",
"MET",
"AFL",
"ALL",
"TEL",
"GWW",
"SRE",
"O",
"AEP",
"IQV",
"JCI",
"AMP",
"FTNT",
"CCI",
"MSCI",
"DLR",
"FAST",
"FIS",
"BK",
"HES",
"STZ",
"IDXX",
"KMB",
"A",
"DOW",
"AME",
"PRU",
"LULU",
"LEN",
"MNST",
"CMI",
"D",
"CTVA",
"ODFL",
"OTIS",
"COR",
"PAYX",
"LHX",
"GIS",
"HUM",
"CNC",
"SYY",
"RSG",
"MLM",
"CSGP",
"PWR",
"IR",
"YUM",
"EXC",
"GEHC",
"FANG",
"IT",
"HAL",
"KR",
"PCG",
"VMC",
"CTSH",
"KMI",
"GEV",
"ACGL",
"MRNA",
"KVUE",
"DG",
"BKR",
"DVN",
"CDW",
"EL",
"ADM",
"GPN",
"PEG",
"PPG",
"VRSK",
"DD",
"RCL",
"MPWR",
"ROK",
"KDP",
"EA",
"EFX",
"EXR",
"DFS",
"ED",
"HIG",
"VICI",
"FICO",
"XYL",
"DAL",
"ANSS",
"XEL",
"BIIB",
"FTV",
"ON",
"KHC",
"HSY",
"WST",
"CBRE",
"MTD",
"KEYS",
"WTW",
"RMD",
"EIX",
"CHTR",
"TSCO",
"CAH",
"WAB",
"EBAY",
"DLTR",
"ZBH",
"LYB",
"TROW",
"AVB",
"HWM",
"TRGP",
"WEC",
"HPQ",
"WY",
"NVR",
"CHD",
"PHM",
"BLDR",
"FITB",
"DOV",
"GLW",
"RJF",
"TTWO",
"BR",
"NDAQ",
"STT",
"WDC",
"MTB",
"HPE",
"AWK",
"IRM",
"SBAC",
"GRMN",
"ALGN",
"DECK",
"DTE",
"STLD",
"ETR",
"HUBB",
"ULTA",
"PTC",
"MOH",
"CPAY",
"NTAP",
"AXON",
"EQR",
"IFF",
"APTV",
"BAX",
"GPC",
"CTRA",
"STE",
"BALL",
"ES",
"ILMN",
"INVH",
"BRO",
"PPL",
"HBAN",
"WAT",
"FE",
"ARE",
"COO",
"TDY",
"LVS",
"CBOE",
"VLTO",
"FSLR",
"CINF",
"AEE",
"TXT",
"MKC",
"RF",
"WBD",
"DRI",
"PFG",
"J",
"OMC",
"NTRS",
"HOLX",
"IEX",
"CLX",
"CNP",
"LH",
"JBL",
"WRB",
"LDOS",
"AVY",
"EXPE",
"SYF",
"DPZ",
"TYL",
"VTR",
"MAS",
"ATO",
"CMS",
"MRO",
"STX",
"EXPD",
"PKG",
"LUV",
"TSN",
"FDS",
"NRG",
"SWKS",
"VRSN",
"TER",
"EG",
"CE",
"CFG",
"AKAM",
"JBHT",
"CCL",
"ENPH",
"ESS",
"BBY",
"SNA",
"TRMB",
"ALB",
"BG",
"EPAM",
"MAA",
"POOL",
"CF",
"ZBRA",
"K",
"EQT",
"CAG",
"SWK",
"NDSN",
"LYV",
"DGX",
"HST",
"KEY",
"UAL",
"VTRS",
"L",
"LKQ",
"WBA",
"PNR",
"DOC",
"IP",
"AMCR",
"KMX",
"RVTY",
"CRL",
"MGM",
"ROL",
"GEN",
"JKHY",
"WRK",
"LNT",
"KIM",
"TAP",
"AES",
"EVRG",
"IPG",
"EMN",
"SJM",
"PODD",
"JNPR",
"ALLE",
"FFIV",
"HII",
"UDR",
"LW",
"QRVO",
"NI",
"CPT",
"TECH",
"APA",
"AOS",
"BBWI",
"MOS",
"UHS",
"CTLT",
"INCY",
"TFX",
"WYNN",
"HRL",
"TPR",
"PAYC",
"NWSA",
"REG",
"DAY",
"AIZ",
"HSIC",
"SOLV",
"MTCH",
"GL",
"BF.B",
"CZR",
"AAL",
"BXP",
"CPB",
"MKTX",
"CHRW",
"PNW",
"GNRC",
"BWA",
"NCLH",
"RHI",
"ETSY",
"FOXA",
"BEN",
"IVZ",
"FMC",
"FRT",
"HAS",
"DVA",
"CMA",
"BIO",
"RL",
"MHK",]

# Diccionario para almacenar los modelos (o el resultado que prefieras)
resultados_modelos = {}

for ticker in lista_tickers:
    print(f"Procesando {ticker}...")
    try:
        # Ejecuta tu función de regresión para el ticker actual
        resultado = regresion(ticker)

        # Almacena el resultado en el diccionario
        resultados_modelos[ticker] = resultado
    except Exception as e:
        print(f"Error al procesar {ticker}: {e}")

# Ahora tienes todos tus resultados almacenados en resultados_modelos


[*********************100%%**********************]  1 of 1 completed
<ipython-input-9-39969a432522>:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  famafrench_df['Date'] = pd.to_datetime(famafrench_df['Date'])
[*********************100%%**********************]  1 of 1 completed

Procesando MSFT...
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.112
Model:                            OLS   Adj. R-squared:                  0.108
Method:                 Least Squares   F-statistic:                     27.81
Date:                Wed, 10 Apr 2024   Prob (F-statistic):           9.03e-22
Time:                        22:16:59   Log-Likelihood:                 2170.5
No. Observations:                 885   AIC:                            -4331.
Df Residuals:                     880   BIC:                            -4307.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const          -0.0058      0.0


<ipython-input-9-39969a432522>:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  famafrench_df['Date'] = pd.to_datetime(famafrench_df['Date'])


                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.117
Model:                            OLS   Adj. R-squared:                  0.113
Method:                 Least Squares   F-statistic:                     29.14
Date:                Wed, 10 Apr 2024   Prob (F-statistic):           8.92e-23
Time:                        22:16:59   Log-Likelihood:                 2148.8
No. Observations:                 885   AIC:                            -4288.
Df Residuals:                     880   BIC:                            -4264.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const          -0.0056      0.001     -7.678      

[*********************100%%**********************]  1 of 1 completed
<ipython-input-9-39969a432522>:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  famafrench_df['Date'] = pd.to_datetime(famafrench_df['Date'])
[*********************100%%**********************]  1 of 1 completed
<ipython-input-9-39969a432522>:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  famafrench_df['Date'] = pd.to_datetime(famafrench_df['Date'])


...

[*********************100%%**********************]  1 of 1 completed
<ipython-input-9-39969a432522>:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  famafrench_df['Date'] = pd.to_datetime(famafrench_df['Date'])


                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.075
Model:                            OLS   Adj. R-squared:                  0.071
Method:                 Least Squares   F-statistic:                     17.87
Date:                Wed, 10 Apr 2024   Prob (F-statistic):           4.06e-14
Time:                        22:21:27   Log-Likelihood:                 1908.0
No. Observations:                 885   AIC:                            -3806.
Df Residuals:                     880   BIC:                            -3782.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const          -0.0063      0.001     -6.608      

[*********************100%%**********************]  1 of 1 completed

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.114
Model:                            OLS   Adj. R-squared:                  0.110
Method:                 Least Squares   F-statistic:                     28.25
Date:                Wed, 10 Apr 2024   Prob (F-statistic):           4.19e-22
Time:                        22:21:28   Log-Likelihood:                 1833.4
No. Observations:                 885   AIC:                            -3657.
Df Residuals:                     880   BIC:                            -3633.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const          -0.0065      0.001     -6.321      


<ipython-input-9-39969a432522>:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  famafrench_df['Date'] = pd.to_datetime(famafrench_df['Date'])


In [ ]:
df_combinado.to_csv('ex.csv', sep = ";", index=True)